# Run the architecture-conformance experiment on a free GPU (Kaggle or Colab)

Your laptop (8 GB RAM, no GPU) cannot run a 7-8B model for the full experiment. This notebook runs the
**same code** on a free cloud GPU with Ollama, then packages the results for download.

> **Status: written but not tested by the person who wrote it.** The author could not reach Ollama or a GPU runtime
> from the environment this was written in, so the install/start steps below follow common practice and may need small
> fixes (every cell prints what it did so problems are visible). The *project code* it calls is covered by the automated test suite (see README).

**Before you start**
1. **Kaggle:** create a notebook, set *Accelerator* to a GPU, turn *Internet* on (needs a verified account). Add `architecture-analyzer-m2-m5.zip` as a dataset or upload it.
   **Colab:** *Runtime > Change runtime type > GPU*, then upload the zip in the Files panel.
2. Check the free-GPU quota and session limits shown by the platform; sessions end, so the **cache** (saved in the results folder) matters.
3. Run the cells top to bottom. Do the **pilot** before the full run.

In [ ]:
# ---- settings you may change -------------------------------------------------
import glob, os
ZIP_PATH   = (glob.glob('/kaggle/input/**/architecture-analyzer-m2-m5.zip', recursive=True)
              + glob.glob('/content/architecture-analyzer-m2-m5.zip')
              + glob.glob('./architecture-analyzer-m2-m5.zip'))
ZIP_PATH   = ZIP_PATH[0] if ZIP_PATH else None
BASE_MODEL = 'qwen2.5:7b-instruct'   # check ollama.com/library for a current 7-8B *instruct* model; name may differ
NUM_CTX    = 4096                    # must exceed the longest prompt (~1,100 tokens) plus the reply
WORK       = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
print('zip:', ZIP_PATH, '| work dir:', WORK)
assert ZIP_PATH, 'Upload architecture-analyzer-m2-m5.zip first (see the markdown above).'

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'NO GPU DETECTED - turn on a GPU accelerator, or expect very slow runs'

## 2. Unpack the project and install its Python dependencies

In [ ]:
import subprocess, os
os.chdir(WORK)
!rm -rf architecture-analyzer && unzip -q -o "{ZIP_PATH}" -d .
os.chdir(f'{WORK}/architecture-analyzer/backend')
!pip install -q -r requirements-dev.txt
!python -m pytest tests -q -x 2>&1 | tail -3

## 3. Install and start Ollama
If the install line fails, read its error: newer installers need `zstd` (`apt-get install -y zstd`), and some
platforms need `sudo`. The server must answer on port 11434 before continuing.

In [ ]:
!apt-get install -y -q zstd > /dev/null 2>&1 || true
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request, os
env = dict(os.environ, OLLAMA_NUM_PARALLEL='1', OLLAMA_CONTEXT_LENGTH=str(NUM_CTX))
server = subprocess.Popen(['ollama', 'serve'], env=env, stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2); print('Ollama is up'); break
    except Exception:
        time.sleep(2)
else:
    print(open('/tmp/ollama.log').read()[-2000:]); raise SystemExit('Ollama did not start')

## 4. Pull the model and give it a bigger context window
Ollama silently truncates prompts longer than its context window. The Modelfile below sets `num_ctx`, and temperature 0
keeps answers repeatable.

In [ ]:
!ollama pull {BASE_MODEL}
open('/tmp/Modelfile', 'w').write(f'FROM {BASE_MODEL}\nPARAMETER num_ctx {NUM_CTX}\nPARAMETER temperature 0\n')
!ollama create arch-judge -f /tmp/Modelfile
!ollama show arch-judge | tee {WORK}/model_info.txt

## 5. Smoke test the model through the project's own client (checks JSON output and speed)

In [ ]:
import os, sys, json
os.environ.update(LLM_BASE_URL='http://localhost:11434/v1', LLM_MODEL='arch-judge', LLM_API_KEY='ollama')
sys.path.insert(0, f'{WORK}/architecture-analyzer/backend')
from analysis.semantic.llm_client import client_from_env
from analysis.semantic.llm_analyzer import parse_verdict_json, REVIEW_VERDICTS
c = client_from_env()
r = c.complete('You are a reviewer. Reply with ONLY JSON.',
               'Reply with {"verdict":"violation","confidence":0.9,"explanation":"test","evidence_used":[]}')
print('ok:', r.ok, '| latency %.1fs' % r.latency_s, '| tokens in/out:', r.input_tokens, r.output_tokens)
print('raw:', r.text[:200]); print('parsed:', parse_verdict_json(r.text, REVIEW_VERDICTS))

## 6. Generate the benchmark and count the calls (no model calls yet)

In [ ]:
%cd {WORK}/architecture-analyzer/backend
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated --seed 7
!python -m experiments.run_experiments --dry-run --seeds 5

## 7. PILOT: one case per scenario (10 cases)
Look at the table, then at `predictions.csv`: is `parsed_ok` true? are `explanation`s sensible? is `latency_s` small?
If most calls fail to parse, fix that **before** spending the full run.

In [ ]:
!python -m experiments.run_experiments --limit-cases 10 --methods evidence_guided,full_hybrid --out ../results/pilot --bootstrap 100
import pandas as pd
p = pd.read_csv('../results/pilot/predictions.csv')
p = p[p.llm_called.astype(bool)]
print('calls:', len(p), '| parsed ok:', int(p.parsed_ok.sum()), '| errors:', int(p.error.notna().sum()),
      '| median latency: %.1fs' % p.latency_s.median())
p[['method','case_id','scenario','truth','verdict','predicted_violation','latency_s','explanation']]

## 8. Dev split (iterate prompts HERE only, never on the test split)
Re-running after a prompt change re-calls only what changed (responses are cached).

In [ ]:
!python -m experiments.run_experiments --splits dev --methods static,full_hybrid,evidence_guided,no_history --out ../results/dev --bootstrap 200

## 9. FULL run, primary setup (freeze prompts and policy first; `docs/ANALYSIS_PLAN.md` says what counts as support)
Without M2 first (99 distinct calls). `--max-calls` stops safely if the session is about to hit a limit; run the same cell again to resume from the cache.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --methods static,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/primary --max-calls 400 --seeds 5

### 9b. M2 (LLM on every edge) — 420 calls, the expensive one. Run it ONCE, on the primary setup.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --methods llm_only --out ../results/primary_m2 --max-calls 600

## 9c. Robustness runs (99 calls each; skip M2)
* **Minimal prompt**: same data, all scenario-specific guidance removed from the system prompts.
* **Held-out wording**: the same repositories with the ADR decision language rewritten. **Commit `docs/ANALYSIS_PLAN.md` to Git before this cell** so the plan provably came first.

In [ ]:
!python -m experiments.run_experiments --splits dev,test --prompt-variant minimal --methods static,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/minimal --max-calls 400 --seeds 5

In [ ]:
!python -m experiments.benchmark.generate --out ../datasets/benchmark/generated_heldout --seed 7 --wording heldout
!python -m experiments.run_experiments --benchmark ../datasets/benchmark/generated_heldout --splits dev,test --methods static,full_hybrid,evidence_guided,no_history,no_intent,generic_selective --out ../results/heldout --max-calls 400 --seeds 5

## 10. Package everything for download

In [ ]:
!cp {WORK}/model_info.txt ../results/primary/ 2>/dev/null || true
!cd .. && zip -qr {WORK}/results.zip results
print('Download', f'{WORK}/results.zip', '(Kaggle: Output tab; Colab: Files panel).')
print(open('../results/primary/summary.md').read()[:3000])